# 🤖 Software Engineering Chatbot — Generalized Conversational RAG (v2)

End-to-end retrieval-augmented generation system specialized in Software Engineering,
upgraded from dataset-FAQ-bot to **robust conversational assistant**.

### Architecture v2 Improvements

| # | What changed | Why it helps |
|---|---|---|
| 1 | `use_top_context_as_answer=False` — LLM always generates | Avoids copying stale FAQ answers; more natural, explanatory responses |
| 2 | Softer thresholds (DEFAULT 0.30) | Reduces unnecessary refusals on valid SE questions |
| 3 | Expert-assistant prompts with own-knowledge fallback | LLM answers from knowledge when retrieval is weak |
| 4 | `intfloat/multilingual-e5-large` embeddings | Stronger retrieval for paraphrases, long queries, FR/EN |
| 5 | BM25 + FAISS hybrid search | BM25 catches exact terms; dense catches semantics |
| 6 | Reranker toggle + evaluation mode | Disable if hurting; analyze impact analytically |
| 7 | Embedding-based domain classification | Replaces rigid keyword list; catches vocabulary variations |
| 8 | Memory extended to 6 turns + topic vector | Richer conversation context |
| 9 | Paraphrase + conversational data augmentation | Training on reformulations → better generalization |
| 10 | FLAN-T5-large with LoRA option | More capacity; LoRA prevents overfitting |
| 11 | Extended evaluation: unseen, paraphrase, OOD | Measures real-world robustness |
| 12 | Natural, explanatory response style | Less FAQ-like, more ChatGPT-like |

### Runtime
- **Settings → Accelerator → GPU T4 x2**, Internet **ON**
- Total: ~20-30 min (larger models + augmentation)


## 1. Setup

In [41]:
# --- 1.1 Install / upgrade -----------------------------------------------
import subprocess, sys, importlib

REQUIRED = {
    "sentence_transformers": "sentence-transformers>=3.0",
    "faiss":                 "faiss-cpu",
    "datasets":              "datasets>=2.14",
    "evaluate":              "evaluate",
    "rouge_score":           "rouge_score",
    "sacrebleu":             "sacrebleu",
    "gradio":                "gradio>=4.0",
    "langdetect":            "langdetect",
    "unidecode":             "unidecode",
    "accelerate":            "accelerate>=0.26",
    # BM25 for hybrid search
    "rank_bm25":             "rank-bm25",
    # LoRA support
    "peft":                  "peft>=0.7",
}

missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("All dependencies ready.")


All dependencies ready.


In [42]:
# --- 1.2 Imports + global config -----------------------------------------
import os, re, json, time, random, warnings, gc, inspect
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DATA_DIR     = WORK_DIR / "data";       DATA_DIR.mkdir(exist_ok=True)
INDEX_DIR    = WORK_DIR / "indexes";    INDEX_DIR.mkdir(exist_ok=True)
MODEL_DIR    = WORK_DIR / "model_ft";   MODEL_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR = WORK_DIR / "artifacts";  ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"CC     : {torch.cuda.get_device_capability(0)}")
print(f"Torch  : {torch.__version__}")
print(f"Work   : {WORK_DIR}")


Device : cuda
GPU    : Tesla T4
CC     : (7, 5)
Torch  : 2.10.0+cu128
Work   : /kaggle/working


In [43]:
# --- 1.3 GPU sanity check -------------------------------------------------
USE_GPU = False
if torch.cuda.is_available():
    try:
        _t = torch.randn(64, 64, device="cuda")
        _ = (_t @ _t).sum().item()
        torch.cuda.synchronize()
        USE_GPU = True
        print("CUDA self-test: OK")
    except Exception as e:
        print(f"CUDA self-test FAILED: {e}")
        USE_GPU = False
print(f"Effective compute: {'GPU' if USE_GPU else 'CPU'}")


CUDA self-test: OK
Effective compute: GPU


## 2. Dataset loading & cleaning

Sources normalized to `(question, reponse, categorie)`.
**Improvement #9**: data augmentation adds paraphrased + conversational variants.


In [44]:
# --- 2.1 Configuration ---------------------------------------------------
MAX_PER_SOURCE = 1500
INCLUDE_GENERAL_QA = False
INCLUDE_UBUNTU_DIALOGUE = False
MIN_LEN, MAX_LEN_Q, MAX_LEN_A = 15, 300, 800

# IMPROVEMENT #2: Softer thresholds — reduced from 1.0 to 0.30.
# The old threshold of 1.0 rejected too many valid SE questions.
# 0.30 allows borderline-domain questions to reach the LLM before refusing.
DEFAULT_THRESHOLD = 0.30
DOMAIN_FALLBACK_MIN_SCORE = -1.0      # was -0.25; now much more permissive
DOMAIN_FALLBACK_MIN_DOMAIN_SCORE = 0.10  # was 0.18; softer domain filter

def basic_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = re.sub(r"<[^>]+>", " ", text)
    t = re.sub(r"```.*?```", " [code] ", t, flags=re.S)
    t = re.sub(r"`[^`]+`", " [code] ", t)
    t = re.sub(r"https?://\S+", " [url] ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def valid_pair(q: str, a: str) -> bool:
    return (MIN_LEN <= len(q) <= MAX_LEN_Q and
            MIN_LEN <= len(a) <= MAX_LEN_A and
            len(a.split()) >= 4)


In [45]:
# --- 2.2 Loaders (one per source) ----------------------------------------
def load_software_engineering_stackexchange(n: int = MAX_PER_SOURCE):
    print("Loading: SE StackExchange ...")
    ds = load_dataset(
        "flax-sentence-embeddings/stackexchange_title_body_jsonl",
        "softwareengineering", split="train", streaming=True,
    )
    rows = []
    for ex in ds:
        raw = basic_clean(ex.get("title_body", ex.get("title", "")))
        if "\n" in raw:
            title, body = raw.split("\n", 1)
            q, a = title.strip(), body.strip()
        else:
            continue
        if valid_pair(q, a):
            rows.append((q, a, "software_engineering"))
        if len(rows) >= n:
            break
    print(f"  -> {len(rows)} rows")
    return rows

def load_stackoverflow(n: int = MAX_PER_SOURCE):
    print("Loading: StackOverflow ...")
    ds = load_dataset(
        "flax-sentence-embeddings/stackexchange_title_body_jsonl",
        "stackoverflow", split="train", streaming=True,
    )
    rows = []
    for ex in ds:
        raw = basic_clean(ex.get("title_body", ""))
        if "\n" not in raw:
            continue
        title, body = raw.split("\n", 1)
        q, a = title.strip(), body.strip()
        if valid_pair(q, a):
            rows.append((q, a, "stackoverflow"))
        if len(rows) >= n:
            break
    print(f"  -> {len(rows)} rows")
    return rows

def load_code_search_net(n: int = MAX_PER_SOURCE):
    print("Loading: CodeSearchNet (python) ...")
    ds = load_dataset("code_search_net", "python",
                      split="train", streaming=True, trust_remote_code=True)
    rows = []
    BAD_PATTERNS = (":param", ":return", ":rtype", ":raises", ":type",
                    ":arg", ":kwarg", ">>>", "Args:", "Returns:", "Raises:",
                    "Parameters\n", "Examples\n", "----")
    for ex in ds:
        doc = basic_clean(ex.get("func_documentation_string", ""))
        name = ex.get("func_name", "") or ""
        if not doc or not name:
            continue
        if any(p in doc for p in BAD_PATTERNS):
            continue
        doc = doc.split("\n")[0].split(". ")[0].strip()
        if not doc.endswith("."):
            doc += "."
        q = f"What does the function '{name}' do?"
        if valid_pair(q, doc) and len(doc.split()) >= 6:
            rows.append((q, doc, "code_search_net"))
        if len(rows) >= n:
            break
    print(f"  -> {len(rows)} rows")
    return rows

def load_squad_v2(n: int = MAX_PER_SOURCE):
    print("Loading: SQuAD v2 ...")
    ds = load_dataset("rajpurkar/squad_v2", split="train", streaming=True)
    rows = []
    for ex in ds:
        q = basic_clean(ex.get("question", ""))
        answers = ex.get("answers", {}).get("text", [])
        if not answers:
            continue
        a = basic_clean(answers[0])
        if valid_pair(q, a):
            rows.append((q, a, "squad"))
        if len(rows) >= n:
            break
    print(f"  -> {len(rows)} rows")
    return rows


In [46]:
# --- 2.3 Curated SE seeds (high-quality anchor, oversampled later) ------
SEED_ROWS = [
    ("What is Agile methodology?",
     "Agile is an iterative software development approach that delivers working software in short cycles called sprints, embraces changing requirements, and emphasizes collaboration between cross-functional teams and customers.",
     "agile"),
    ("What is Scrum?",
     "Scrum is an Agile framework with fixed-length sprints, three roles (Product Owner, Scrum Master, Development Team), and ceremonies such as sprint planning, daily standup, sprint review and retrospective.",
     "agile"),
    ("What is a user story?",
     "A user story is a short, informal description of a feature from the user's perspective, typically written as: As a <role>, I want <goal> so that <benefit>.",
     "agile"),
    ("What is a sprint retrospective?",
     "A sprint retrospective is a Scrum ceremony at the end of each sprint where the team inspects how the sprint went and identifies improvements for the next one.",
     "agile"),
    ("What is UML?",
     "UML (Unified Modeling Language) is a standardized graphical modeling language used to specify, visualize and document object-oriented software systems. It defines about 14 diagram types split into structural and behavioral diagrams.",
     "uml"),
    ("What is a class diagram?",
     "A class diagram is a UML structural diagram showing the classes of a system, their attributes, methods, and the static relationships between them.",
     "uml"),
    ("What is a sequence diagram?",
     "A sequence diagram is a UML behavioral diagram that shows how objects interact over time through messages exchanged between lifelines.",
     "uml"),
    ("What is the difference between aggregation and composition in UML?",
     "Both express a whole-part relationship. Aggregation is weak: the part can exist without the whole. Composition is strong: the part's lifecycle is bound to the whole, so destroying the whole destroys the parts.",
     "uml"),
    ("What is a use case diagram?",
     "A use case diagram is a UML behavioral diagram representing functional requirements through actors and use cases, connected by associations.",
     "uml"),
    ("What is object-oriented programming?",
     "OOP is a paradigm built around objects that bundle data and behavior. Its four pillars are encapsulation, inheritance, polymorphism and abstraction.",
     "oop"),
    ("What is encapsulation?",
     "Encapsulation is the OOP principle of bundling data and the methods that operate on it inside a class, and restricting direct access to internal state through visibility modifiers.",
     "oop"),
    ("What is polymorphism?",
     "Polymorphism is the OOP ability for one interface or method name to behave differently depending on the actual object type, through overriding or overloading.",
     "oop"),
    ("What is inheritance?",
     "Inheritance is an OOP mechanism by which a subclass derives attributes and methods from a parent class, supporting code reuse and is-a relationships.",
     "oop"),
    ("What is an abstract class?",
     "An abstract class is a class that cannot be instantiated directly and is meant to be subclassed. It typically declares abstract methods that subclasses must implement.",
     "oop"),
    ("What are SOLID principles?",
     "SOLID is five OOP design principles: Single Responsibility, Open/Closed, Liskov Substitution, Interface Segregation and Dependency Inversion. Together they promote maintainable, extensible code.",
     "oop"),
    ("What is the difference between an interface and an abstract class?",
     "An interface declares only method signatures and a class can implement many of them. An abstract class can hold state and concrete methods, and a class can extend only one of them.",
     "oop"),
    ("What is a design pattern?",
     "A design pattern is a reusable, named solution to a recurring problem in software design. The GoF catalog groups them into creational, structural and behavioral categories.",
     "patterns"),
    ("What is the Singleton design pattern?",
     "Singleton is a creational pattern ensuring a class has at most one instance and providing a global access point to it. Useful for shared resources like configuration or logging.",
     "patterns"),
    ("What is the Observer design pattern?",
     "Observer is a behavioral pattern where a subject maintains a list of dependents (observers) and notifies them automatically of state changes.",
     "patterns"),
    ("What is the Factory design pattern?",
     "Factory is a creational pattern that delegates object creation to a separate method or class, decoupling the calling code from concrete classes.",
     "patterns"),
    ("What is the Strategy design pattern?",
     "Strategy defines a family of interchangeable algorithms, encapsulates each one, and lets the client choose which to use at runtime, favoring composition over inheritance.",
     "patterns"),
    ("What is the MVC pattern?",
     "MVC (Model-View-Controller) is an architectural pattern separating data and business logic (Model), user interface (View), and input/coordination logic (Controller).",
     "patterns"),
    ("What is Git?",
     "Git is a distributed version control system that tracks changes, supports branching and merging, and lets every developer hold a full copy of the project history locally.",
     "git"),
    ("What is a Git branch?",
     "A Git branch is a lightweight movable pointer to a commit, letting developers work on features or fixes in isolation before merging back to the main line.",
     "git"),
    ("What is the difference between git merge and git rebase?",
     "Both integrate changes between branches. git merge creates a merge commit preserving history. git rebase rewrites commits onto a new base for a linear history, changing commit hashes.",
     "git"),
    ("What is a pull request?",
     "A pull request is a code review mechanism on platforms like GitHub or GitLab where a developer proposes changes from one branch to another for discussion and approval.",
     "git"),
    ("What is unit testing?",
     "Unit testing is writing automated tests that verify the smallest pieces of code (functions, methods) in isolation, usually mocking external dependencies.",
     "testing"),
    ("What is integration testing?",
     "Integration testing verifies that several components work correctly together, exercising interactions between modules, services, databases or external APIs.",
     "testing"),
    ("What is Test-Driven Development?",
     "TDD is a discipline summarized as red-green-refactor: write a failing test, write the minimum code to pass it, then refactor while keeping tests green.",
     "testing"),
    ("What is the difference between mocks and stubs?",
     "Stubs return canned answers to calls during a test. Mocks are pre-programmed with expectations about which calls they should receive and can fail the test if those expectations aren't met.",
     "testing"),
    ("What is code coverage?",
     "Code coverage is a metric measuring the proportion of source code executed by a test suite, typically as line, branch or statement coverage.",
     "testing"),
    ("What is a REST API?",
     "A REST API is an HTTP-based interface exposing resources via URLs, manipulated through standard verbs (GET, POST, PUT, PATCH, DELETE), stateless between requests, typically exchanging JSON.",
     "api"),
    ("What is the difference between REST and GraphQL?",
     "REST exposes multiple endpoints with fixed response shapes. GraphQL exposes a single endpoint where the client queries exactly the fields it needs, reducing over-fetching but adding server complexity.",
     "api"),
    ("What is an idempotent HTTP method?",
     "An HTTP method is idempotent when calling it multiple times has the same effect as calling it once. GET, PUT and DELETE are idempotent; POST generally is not.",
     "api"),
    ("What is JWT authentication?",
     "JWT (JSON Web Token) is a compact signed token format for stateless authentication. The server issues a token after login; the client sends it in the Authorization header on subsequent requests.",
     "api"),
    ("What is CI/CD?",
     "CI/CD stands for Continuous Integration (automatically building and testing every change) and Continuous Delivery or Deployment (automatically releasing changes to staging or production).",
     "devops"),
    ("What is Docker?",
     "Docker packages an application and its dependencies into a lightweight, portable container that runs identically on any host with a compatible runtime.",
     "devops"),
    ("What is the difference between a container and a virtual machine?",
     "A virtual machine emulates an entire OS on top of a hypervisor. A container shares the host kernel and isolates only user space, making it lighter, faster to start and more density-friendly.",
     "devops"),
    ("What is Kubernetes?",
     "Kubernetes is an open-source container orchestrator that automates the deployment, scaling, networking and self-healing of containerized applications across a cluster.",
     "devops"),
    ("What is DevOps?",
     "DevOps is a set of cultural practices and tools that bridge software development and operations, aiming to shorten the delivery cycle through automation, collaboration and continuous integration and delivery.",
     "devops"),
    ("What is a relational database?",
     "A relational database organizes data into tables of rows and columns with relationships expressed through foreign keys, queried using SQL. Examples include PostgreSQL, MySQL and SQLite.",
     "db"),
    ("What is database normalization?",
     "Normalization structures a relational schema to reduce redundancy and avoid update anomalies, typically by decomposing tables and applying normal forms (1NF, 2NF, 3NF, BCNF).",
     "db"),
    ("What is the difference between SQL and NoSQL?",
     "SQL databases enforce a fixed relational schema and ACID transactions. NoSQL databases (document, key-value, column, graph) trade strict schemas for flexibility and horizontal scalability.",
     "db"),
    ("What is an index in a database?",
     "An index is a data structure (commonly a B-tree) that speeds up read queries on a column at the cost of slower writes and extra storage.",
     "db"),
    ("What is ACID in databases?",
     "ACID stands for Atomicity, Consistency, Isolation and Durability - four properties that guarantee reliable processing of database transactions.",
     "db"),
    ("What is software architecture?",
     "Software architecture is the set of high-level structural decisions about a system: its components, their responsibilities, their interactions, and the constraints that shape them.",
     "architecture"),
    ("What is the difference between monolithic and microservices architecture?",
     "A monolith deploys all functionality as one unit. Microservices split the system into small, independently deployable services communicating over the network, trading operational simplicity for scalability.",
     "architecture"),
    ("What is coupling and cohesion?",
     "Coupling measures how strongly modules depend on each other - we aim for low coupling. Cohesion measures how closely the responsibilities inside a module belong together - we aim for high cohesion.",
     "architecture"),
    ("What is low coupling?",
     "Low coupling means modules have few dependencies on each other, so a change in one module has limited impact on the others.",
     "architecture"),
    ("What is high cohesion?",
     "High cohesion means the responsibilities inside a module are closely related and focused on a single well-defined purpose.",
     "architecture"),
    ("What is load balancing?",
     "Load balancing distributes incoming traffic across multiple servers to improve responsiveness and availability, using strategies like round-robin, least connections or hashing.",
     "architecture"),
    ("What is caching?",
     "Caching stores the result of expensive computations or remote calls in fast storage so subsequent identical requests are served quickly, at the cost of staleness and invalidation complexity.",
     "architecture"),
    ("What is dependency injection?",
     "Dependency injection is a design technique where an object receives its dependencies from the outside instead of creating them, improving testability and decoupling.",
     "patterns"),
]

print(f"Seed rows: {len(SEED_ROWS)}")


Seed rows: 53


In [47]:
# --- 2.4 IMPROVEMENT #9: Data augmentation with paraphrases + conversational forms ---
# Original notebook had no augmentation — the model only saw FAQ-style questions.
# Adding paraphrased + conversational + long-form variants helps the model generalize
# to how real users actually phrase questions.

CONVERSATIONAL_PREFIXES = [
    "Can you explain {}?",
    "I'm trying to understand {}. Could you help?",
    "What exactly is {}?",
    "Could you walk me through {}?",
    "I keep hearing about {} — what does it mean?",
    "Help me understand {}.",
    "In simple terms, what is {}?",
]

SCENARIO_TEMPLATES = [
    ("How do I apply {} in a real project?",
     "To apply {} in a real project, understand its intent and constraints first, then gradually introduce it where the problem occurs. Start small, measure impact, and refactor incrementally."),
    ("When should I use {}?",
     "Use {} when the problem it solves appears in your codebase. Avoid applying it prematurely; patterns add structure but also complexity."),
    ("What are the benefits of {}?",
     "The main benefits of {} include improved maintainability, clearer separation of concerns, and easier testing. The tradeoff is added abstraction and learning curve."),
]

def augment_with_paraphrases(rows: List[Tuple]) -> List[Tuple]:
    """Generate conversational and paraphrase variants of seed rows."""
    augmented = []
    for q, a, cat in rows:
        # Extract the core concept (rough heuristic: remove 'What is', 'What are', etc.)
        concept = re.sub(r'^(what is |what are |define |explain )', '', q.lower()).rstrip('?').strip()
        if len(concept) < 3 or len(concept) > 60:
            continue
        # Add a few conversational variants
        for tmpl in random.sample(CONVERSATIONAL_PREFIXES, min(2, len(CONVERSATIONAL_PREFIXES))):
            new_q = tmpl.format(concept)
            if valid_pair(new_q, a):
                augmented.append((new_q, a, cat))
        # Add one scenario variant
        tmpl_q, tmpl_a = random.choice(SCENARIO_TEMPLATES)
        new_q2 = tmpl_q.format(concept)
        new_a2 = tmpl_a.format(concept)
        if valid_pair(new_q2, new_a2):
            augmented.append((new_q2, new_a2, cat))
    return augmented

augmented_rows = augment_with_paraphrases(SEED_ROWS)
print(f"Augmented rows generated: {len(augmented_rows)}")
print("Examples:")
for q, a, c in augmented_rows[:4]:
    print(f"  [{c}] {q}")


Augmented rows generated: 156
Examples:
  [agile] Help me understand agile methodology.
  [agile] Can you explain agile methodology?
  [agile] How do I apply agile methodology in a real project?
  [agile] Help me understand scrum.


In [48]:
# --- 2.5 Run all loaders and unify into a single DataFrame ---------------
all_rows = list(SEED_ROWS) + augmented_rows  # include augmentations

loaders = [
    load_software_engineering_stackexchange,
    load_stackoverflow,
    load_code_search_net,
]
if INCLUDE_GENERAL_QA:
    loaders.append(load_squad_v2)

print(f"General QA enabled     : {INCLUDE_GENERAL_QA}")
print(f"Ubuntu dialogue enabled: {INCLUDE_UBUNTU_DIALOGUE}")

for loader in loaders:
    try:
        all_rows.extend(loader())
    except Exception as e:
        print(f"  loader {loader.__name__} failed: {e}")

df = pd.DataFrame(all_rows, columns=["question", "reponse", "categorie"])
df = df.drop_duplicates(subset=["question"]).reset_index(drop=True)

print(f"\nTotal rows after dedup : {len(df)}")
print("Per category:")
print(df["categorie"].value_counts())

df.to_csv(DATA_DIR / "se_qa_dataset_raw.csv", index=False)
print(f"\nSaved raw dataset to {DATA_DIR / 'se_qa_dataset_raw.csv'}")


General QA enabled     : False
Ubuntu dialogue enabled: False
Loading: SE StackExchange ...


Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/92 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


  loader load_software_engineering_stackexchange failed: BuilderConfig 'softwareengineering' not found. Available: ['default']
Loading: StackOverflow ...


Resolving data files:   0%|          | 0/92 [00:00<?, ?it/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  loader load_stackoverflow failed: BuilderConfig 'stackoverflow' not found. Available: ['default']
Loading: CodeSearchNet (python) ...
  -> 1500 rows

Total rows after dedup : 1683
Per category:
categorie
code_search_net    1474
patterns             28
oop                  28
architecture         25
uml                  20
db                   20
testing              20
devops               20
agile                16
api                  16
git                  16
Name: count, dtype: int64

Saved raw dataset to /kaggle/working/data/se_qa_dataset_raw.csv


In [49]:
# --- 2.6 Rebalance: cap per-category + oversample seeds ------------------
MAX_PER_CATEGORY = 600

balanced_parts = []
for cat, group in df.groupby("categorie"):
    if len(group) > MAX_PER_CATEGORY:
        balanced_parts.append(group.sample(MAX_PER_CATEGORY, random_state=SEED))
    else:
        balanced_parts.append(group)
df_balanced = pd.concat(balanced_parts, ignore_index=True)

SEED_CATS = {"agile", "uml", "oop", "patterns", "git", "testing",
             "api", "devops", "db", "architecture"}
seed_rows_df = df_balanced[df_balanced["categorie"].isin(SEED_CATS)]

df = pd.concat([df_balanced] + [seed_rows_df] * 4, ignore_index=True)
df = df.drop_duplicates(subset=["question"]).reset_index(drop=True)

print(f"Rebalanced corpus: {len(df)} rows")
print("\nPer category:")
print(df["categorie"].value_counts())

df.to_csv(DATA_DIR / "se_qa_dataset.csv", index=False)
print(f"\nSaved final dataset to {DATA_DIR / 'se_qa_dataset.csv'}")


Rebalanced corpus: 809 rows

Per category:
categorie
code_search_net    600
oop                 28
patterns            28
architecture        25
devops              20
testing             20
db                  20
uml                 20
agile               16
api                 16
git                 16
Name: count, dtype: int64

Saved final dataset to /kaggle/working/data/se_qa_dataset.csv


## 3. Preprocessing

In [50]:
# --- 3.1 Text cleaner + retrieval-key normalizer -------------------------
from unidecode import unidecode

_PUNCT_RE = re.compile(r"[^\w\s\?\!\.\,\-:;/]")
_WS_RE    = re.compile(r"\s+")

def normalize_for_retrieval(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = unidecode(text).lower()
    t = _PUNCT_RE.sub(" ", t)
    t = _WS_RE.sub(" ", t).strip()
    return t

df["question_norm"] = df["question"].apply(normalize_for_retrieval)
df = df[df["question_norm"].str.len() > 0].reset_index(drop=True)

print(f"Final corpus rows: {len(df)}")
df[["question", "question_norm", "categorie"]].head(3)


Final corpus rows: 809


,question,question_norm,categorie
0,What is Agile methodology?,what is agile methodology?,agile
1,What is Scrum?,what is scrum?,agile
2,What is a user story?,what is a user story?,agile


## 4. Train / validation split

In [51]:
# --- 4.1 Safe stratified split -------------------------------------------
from sklearn.model_selection import train_test_split

category_counts = df["categorie"].value_counts()
can_stratify = df["categorie"].nunique() > 1 and category_counts.min() >= 2

if not can_stratify:
    rare = category_counts[category_counts < 2]
    print(f"Stratification disabled: categories with <2 rows -> {rare.to_dict()}")

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=SEED,
    stratify=df["categorie"] if can_stratify else None,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")


Train : 728
Val   : 81


## 5. FLAN-T5 fine-tuning

**Improvement #10**: Upgraded prompt prefix to encourage explanation rather than copying.
**Improvement #1**: Training prepares the model to *explain*, not *retrieve*.
LoRA option is included as a commented block for hardware-constrained environments.


In [52]:
# --- 5.1 Tokenize datasets (with target quality filter) ------------------
# IMPROVEMENT #10: Using flan-t5-large for more capacity.
# Switch back to "google/flan-t5-base" if GPU memory is insufficient.
# Revenir à base — large OOM même avec gradient checkpointing sur T4 x2
GEN_MODEL_NAME = "google/flan-t5-base"
MAX_INPUT_LEN  = 384
MAX_TARGET_LEN = 256

PROMPT_PREFIX = (
    "You are an expert software engineering assistant. "
    "Provide a clear, educational, and technically accurate explanation: "
)

print(f"Loading tokenizer: {GEN_MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)

BAD_TARGET_PATTERNS = (":param", ":return", ":rtype", ":raises", ":type",
                       "```", "def ", "import ", ">>>", "Args:", "Returns:")

def is_good_target(text: str) -> bool:
    if not isinstance(text, str):
        return False
    if any(p in text for p in BAD_TARGET_PATTERNS):
        return False
    if len(text.split()) < 5:
        return False
    return True

before = len(train_df)
train_df_ft = train_df[train_df["reponse"].apply(is_good_target)].reset_index(drop=True)
val_df_ft   = val_df[val_df["reponse"].apply(is_good_target)].reset_index(drop=True)
print(f"After quality filter: train {before} -> {len(train_df_ft)}, val {len(val_df)} -> {len(val_df_ft)}")

def tokenize_row(batch):
    inputs  = [PROMPT_PREFIX + q for q in batch["question"]]
    targets = batch["reponse"]
    model_in = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True)
    labels   = tokenizer(text_target=targets, max_length=MAX_TARGET_LEN, truncation=True)
    model_in["labels"] = labels["input_ids"]
    return model_in

train_ds = Dataset.from_pandas(train_df_ft[["question", "reponse"]])
val_ds   = Dataset.from_pandas(val_df_ft[["question", "reponse"]])

train_tok = train_ds.map(tokenize_row, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_row,   batched=True, remove_columns=val_ds.column_names)

print(f"Tokenized train : {len(train_tok)}")
print(f"Tokenized val   : {len(val_tok)}")


Loading tokenizer: google/flan-t5-base
After quality filter: train 728 -> 725, val 81 -> 81


Map:   0%|          | 0/725 [00:00<?, ? examples/s]

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

Tokenized train : 725
Tokenized val   : 81


In [53]:
# --- 5.2 Fine-tune avec LoRA sur un seul GPU ----------------------------
# POURQUOI UN SEUL GPU : DataParallel (T4 x2) double le buffer gradient
# pendant le backward (ReduceAddCoalesced) — c'est exactement ce que montre
# la traceback. Forcer CUDA_VISIBLE_DEVICES="0" supprime ce doublement.
#
# POURQUOI LoRA : même sur un GPU, flan-t5-large consomme ~11 GB rien que
# pour les poids + optimizer states. LoRA gèle 98% des paramètres et
# n'entraîne que ~4 M params au lieu de 770 M → tient dans 14.5 GB.
# Avec flan-t5-base + LoRA on reste largement sous les 6 GB.

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # un seul GPU
os.environ["PYTORCH_ALLOC_CONF"]   = "expandable_segments:True"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

from peft import get_peft_model, LoraConfig, TaskType

print(f"Loading model: {GEN_MODEL_NAME}")
model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)
model.gradient_checkpointing_enable()
model.config.use_cache = False

lora_config = LoraConfig(
    task_type     = TaskType.SEQ_2_SEQ_LM,
    r             = 16,
    lora_alpha    = 32,
    lora_dropout  = 0.05,
    target_modules= ["q", "v"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # devrait afficher ~1-4 M params

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding="longest")

args = Seq2SeqTrainingArguments(
    output_dir                  = str(MODEL_DIR),
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 4,   # effective batch = 16
    learning_rate               = 3e-4,   # LoRA tolère un LR plus élevé
    weight_decay                = 0.01,
    warmup_steps                = 200,
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 1,
    fp16                        = True,
    predict_with_generate       = True,
    generation_max_length       = MAX_TARGET_LEN,
    report_to                   = "none",
    seed                        = SEED,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 2,
)

trainer_kwargs = dict(
    model         = model,
    args          = args,
    train_dataset = train_tok,
    eval_dataset  = val_tok,
    data_collator = data_collator,
)
trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)
trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=2))

allocated = torch.cuda.memory_allocated() / 1e9
print(f"VRAM avant training : {allocated:.2f} GB")
print("Starting fine-tuning ...")
t0 = time.time()
trainer.train()
print(f"Fine-tuning done in {(time.time()-t0)/60:.1f} min")

# Fusionner les poids LoRA avant de sauvegarder (optionnel mais recommandé)
model = model.merge_and_unload()
model.save_pretrained(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Saved to {MODEL_DIR / 'final'}")

Loading model: google/flan-t5-base


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096
VRAM avant training : 5.20 GB
Starting fine-tuning ...


Epoch,Training Loss,Validation Loss
1,No log,19.655699
2,No log,19.605991
3,78.127056,18.726061


Fine-tuning done in 3.3 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /kaggle/working/model_ft/final


In [54]:
# --- 5.3 Reload the fine-tuned model + free trainer memory ---------------
gen_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR / "final"))
gen_model     = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR / "final")).to(DEVICE)
gen_model.eval()

del trainer, model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("Fine-tuned generator ready.")


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Fine-tuned generator ready.


## 6. Embeddings + FAISS Indexing

**Improvement #4**: Replaced `paraphrase-multilingual-MiniLM-L12-v2` (384d) with
`intfloat/multilingual-e5-large` (1024d). E5-large is trained with instruction-based
contrastive learning, giving much stronger retrieval for paraphrases and reformulations.

**Note**: If GPU memory is constrained, use `intfloat/multilingual-e5-base` instead.


In [55]:
# --- 6.1 Load stronger multilingual bi-encoder --------------------------
# IMPROVEMENT #4: intfloat/multilingual-e5-large replaces paraphrase-multilingual-MiniLM.
# E5-large uses instruction prefixes ("query: " / "passage: ") for better retrieval.
# It achieves ~15-20% better recall on multilingual retrieval benchmarks (MIRACL, MTEB).
#
# Fallback: "intfloat/multilingual-e5-base" if memory is tight.
# For English-only: "BAAI/bge-large-en-v1.5"

EMBED_MODEL_NAME = "intfloat/multilingual-e5-large"
print(f"Loading encoder: {EMBED_MODEL_NAME}")
encoder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print(f"Embedding dim: {encoder.get_sentence_embedding_dimension()}")

def embed(texts: List[str], batch_size: int = 64, is_query: bool = False) -> np.ndarray:
    """
    Encode texts with E5 instruction prefix.
    E5 requires 'query: ' prefix for queries and 'passage: ' for passages.
    This distinction significantly improves retrieval quality.
    """
    # Add E5 instruction prefix
    prefix = "query: " if is_query else "passage: "
    prefixed = [prefix + t for t in texts]
    return encoder.encode(
        prefixed,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # L2-normalize for cosine via dot product
    ).astype("float32")

print("Encoder ready. Note: E5 uses 'query:'/'passage:' instruction prefixes.")


Loading encoder: intfloat/multilingual-e5-large


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 1024
Encoder ready. Note: E5 uses 'query:'/'passage:' instruction prefixes.


In [56]:
# --- 6.2 Encode corpus + build FAISS index -------------------------------
# Corpus questions are encoded as "passage: <text>" (E5 convention)
corpus_vecs = embed(df["question_norm"].tolist(), is_query=False)
print(f"Corpus embeddings shape: {corpus_vecs.shape}")

dim = corpus_vecs.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(corpus_vecs)
print(f"FAISS index: ntotal={faiss_index.ntotal}, dim={faiss_index.d}")

np.save(ARTIFACT_DIR / "corpus_vecs.npy", corpus_vecs)
df.to_parquet(ARTIFACT_DIR / "corpus.parquet")
faiss.write_index(faiss_index, str(INDEX_DIR / "corpus.faiss"))
print("FAISS artifacts saved.")


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Corpus embeddings shape: (809, 1024)
FAISS index: ntotal=809, dim=1024
FAISS artifacts saved.


## 7. Hybrid Search (BM25 + FAISS)

**Improvement #5**: Added BM25 sparse retrieval alongside FAISS dense retrieval.
Dense search excels at semantic similarity; BM25 excels at exact keyword/term matching.
Combining both (hybrid search) handles paraphrases *and* precise technical terms.


In [57]:
# --- 7.1 Build BM25 index ------------------------------------------------
# BM25 (Okapi BM25) is a classical sparse retrieval model based on term frequency.
# It catches exact technical terms that dense models may miss after normalization.
print("Building BM25 index ...")
tokenized_corpus = [doc.split() for doc in df["question_norm"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index built on {len(tokenized_corpus)} documents.")


Building BM25 index ...
BM25 index built on 809 documents.


In [58]:
# --- 7.2 Hybrid search function ------------------------------------------
def search_topk_dense(query: str, k: int = 20) -> List[Dict]:
    """Dense FAISS retrieval. Uses 'query: ' E5 prefix for queries."""
    
    q_vec = embed([normalize_for_retrieval(query)], is_query=True)
    scores, idxs = faiss_index.search(q_vec, k)

    out = []
    for score, i in zip(scores[0], idxs[0]):
        if i < 0:
            continue

        row = df.iloc[int(i)]

        out.append({
            "score": float(score),
            "question": row["question"],
            "reponse": row["reponse"],
            "categorie": row["categorie"],
            "idx": int(i),
        })

    return out


def search_topk_bm25(query: str, k: int = 20) -> List[Dict]:
    """Sparse BM25 retrieval — strong for exact technical terms."""

    tokens = normalize_for_retrieval(query).split()
    bm25_scores = bm25.get_scores(tokens)

    top_idxs = np.argsort(bm25_scores)[::-1][:k]

    out = []
    for i in top_idxs:
        if bm25_scores[i] <= 0:
            continue

        row = df.iloc[int(i)]

        out.append({
            "bm25_score": float(bm25_scores[i]),
            "question": row["question"],
            "reponse": row["reponse"],
            "categorie": row["categorie"],
            "idx": int(i),
        })

    return out


def hybrid_search(query: str, k: int = 25, alpha: float = 0.6) -> List[Dict]:
    """
    Hybrid retrieval combining dense (FAISS/E5) and sparse (BM25) scores.
    alpha controls the blend: higher alpha = more weight on dense retrieval.
    """

    dense_results = search_topk_dense(query, k=k)
    bm25_results = search_topk_bm25(query, k=k)

    # Normalize BM25 scores
    if bm25_results:
        max_bm25 = max(r["bm25_score"] for r in bm25_results) or 1.0
        bm25_map = {
            r["idx"]: r["bm25_score"] / max_bm25
            for r in bm25_results
        }
    else:
        bm25_map = {}

    merged: Dict[int, Dict] = {}

    for r in dense_results:
        r["bm25_norm"] = bm25_map.get(r["idx"], 0.0)
        r["hybrid_score"] = (
            alpha * r["score"]
            + (1 - alpha) * r["bm25_norm"]
        )

        merged[r["idx"]] = r

    # Add BM25-only results
    for r in bm25_results:
        if r["idx"] not in merged:
            r["score"] = 0.0
            r["bm25_norm"] = (
                r["bm25_score"] / max_bm25
                if bm25_results else 1.0
            )

            r["hybrid_score"] = (
                (1 - alpha) * r["bm25_norm"]
            )

            merged[r["idx"]] = r

    ranked = sorted(
        merged.values(),
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    return ranked[:k]


# Sanity check
print("Hybrid search test:")

for h in hybrid_search("How does git rebase work?", k=3):
    print(
        f"  [dense={h['score']:.3f} "
        f"bm25={h['bm25_norm']:.3f} "
        f"hybrid={h['hybrid_score']:.3f}] "
        f"{h['question'][:70]}"
    )

Hybrid search test:


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  [dense=0.892 bm25=0.000 hybrid=0.535] What exactly is the difference between git merge and git rebase?
  [dense=0.890 bm25=0.000 hybrid=0.534] What is the difference between git merge and git rebase?
  [dense=0.888 bm25=0.000 hybrid=0.533] In simple terms, what is the difference between git merge and git reba


## 8. Reranking with CrossEncoder

**Improvement #6**: Added reranker toggle + evaluation mode.
The CrossEncoder can sometimes hurt if its domain calibration doesn't match SE Q&A.
Now you can disable it or run in evaluation mode to measure its actual impact.


In [59]:
# --- 8.1 Load CrossEncoder + configure reranker toggle ------------------
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# IMPROVEMENT #6: Reranker toggle.
# Set USE_RERANKER=False to disable and compare retrieval quality directly.
USE_RERANKER = True
RERANKER_MIN_SCORE = -5.0  # if top CE score < this, fall back to hybrid scores

print(f"Loading reranker: {RERANKER_NAME}")
reranker = CrossEncoder(RERANKER_NAME, device=DEVICE)
print(f"Reranker enabled: {USE_RERANKER}")
print(f"Reranker min score threshold: {RERANKER_MIN_SCORE}")


Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker enabled: True
Reranker min score threshold: -5.0


In [60]:
# --- 8.2 Query expansion + IMPROVEMENT #7: Semantic domain classification ---
from unidecode import unidecode

TERM_ALIASES = {
    "genie logiciel": "software engineering",
    "architecture logicielle": "software architecture",
    "patron de conception": "design pattern",
    "patrons de conception": "design patterns",
    "diagramme de classes": "class diagram",
    "diagramme de sequence": "sequence diagram",
    "diagramme de cas d utilisation": "use case diagram",
    "base de donnees": "database",
    "api rest": "rest api",
    "integration continue": "continuous integration",
    "livraison continue": "continuous delivery",
    "deploiement continu": "continuous deployment",
    "test unitaire": "unit test",
    "tests unitaires": "unit tests",
    "test d integration": "integration test",
    "couplage faible": "low coupling",
    "forte cohesion": "high cohesion",
    "couplage": "coupling",
    "cohesion": "cohesion",
    "poo": "object oriented programming",
    "solid": "single responsibility open closed liskov interface segregation dependency inversion",
    "ci cd": "continuous integration continuous delivery",
    "pull request": "code review branch merge request",
    "merge request": "pull request code review branch merge",
    "microservices": "microservices distributed systems services",
    "docker": "containers containerization",
    "kubernetes": "container orchestration deployment scaling",
}
STOPWORDS = {
    "what", "is", "the", "a", "an", "and", "or", "of", "to", "in", "for", "on",
    "difference", "between", "how", "does", "do", "i", "we", "you", "are", "que",
    "quoi", "est", "les", "des", "une", "un", "du", "de", "la", "le", "et", "dans",
    "entre", "comment", "pourquoi", "quels", "quelle", "quelles", "explique", "moi",
    "with", "without", "from", "into", "about", "when", "why", "can", "should",
}

# IMPROVEMENT #7: SE domain anchor sentences for semantic classification.
# Instead of checking keyword presence, we compute cosine similarity to these
# representative sentences. This handles vocabulary variations gracefully.
SE_DOMAIN_ANCHORS = [
    "software engineering design architecture",
    "programming code development testing",
    "database api web services deployment",
    "agile scrum git version control",
    "object oriented design patterns solid principles",
]
_anchor_vecs = None   # lazy-loaded on first use

def get_anchor_vecs() -> np.ndarray:
    global _anchor_vecs
    if _anchor_vecs is None:
        _anchor_vecs = embed(SE_DOMAIN_ANCHORS, is_query=False)
    return _anchor_vecs

def semantic_domain_score(text: str) -> float:
    """
    IMPROVEMENT #7: Semantic domain classification.
    Returns cosine similarity to SE domain anchor sentences.
    Replaces brittle keyword matching with embedding-based routing.
    """
    if not text or not text.strip():
        return 0.0
    q_vec = embed([normalize_for_retrieval(text)], is_query=True)
    anchors = get_anchor_vecs()
    sims = np.dot(q_vec, anchors.T)[0]   # shape (n_anchors,)
    return float(np.max(sims))

def expand_query(text: str) -> str:
    norm = normalize_for_retrieval(text)
    expanded = norm
    for src, tgt in TERM_ALIASES.items():
        if src in expanded and tgt not in expanded:
            expanded = expanded.replace(src, f"{src} {tgt}")
    return expanded

def keyword_set(text: str) -> set:
    return {
        tok for tok in expand_query(text).split()
        if len(tok) > 2 and tok not in STOPWORDS and not tok.isdigit()
    }

def keyword_overlap(query: str, candidate_question: str) -> float:
    q_terms = keyword_set(query)
    c_terms = keyword_set(candidate_question)
    if not q_terms or not c_terms:
        return 0.0
    return len(q_terms & c_terms) / len(q_terms)

def build_query_variants(text: str) -> List[str]:
    variants = [text]
    expanded = expand_query(text)
    if expanded != normalize_for_retrieval(text):
        variants.append(expanded)
    norm = normalize_for_retrieval(text)
    replacements = {
        " vs ": " difference between ",
        " versus ": " difference between ",
        " c est quoi ": " what is ",
        " qu est ce que ": " what is ",
        " explique moi ": " explain ",
        " comment ": " how ",
        " pourquoi ": " why ",
    }
    rewritten = f" {norm} "
    for src, tgt in replacements.items():
        rewritten = rewritten.replace(src, f" {tgt} ")
    rewritten = " ".join(rewritten.split())
    if rewritten and rewritten not in variants:
        variants.append(rewritten)
    terms = sorted(keyword_set(text))
    if terms:
        variants.append(" ".join(terms))
    deduped, seen = [], set()
    for item in variants:
        key = normalize_for_retrieval(item)
        if key and key not in seen:
            deduped.append(item); seen.add(key)
    return deduped


In [61]:
# --- 8.3 Retrieve + rerank (with reranker toggle + fallback) ----------
def retrieve_and_rerank(
    query: str,
    alternate_queries: Optional[List[str]] = None,
    k_retrieve: int = 30,
    k_final: int = 5,
) -> List[Dict]:
    """
    Full pipeline: hybrid search → optional CrossEncoder reranking → score fusion.
    IMPROVEMENT #5+#6: Hybrid search + togglable reranker.
    """
    raw_queries = [query] + [q for q in (alternate_queries or []) if q and q != query]
    query_variants = []
    for q in raw_queries:
        query_variants.extend(build_query_variants(q))

    # Step 1: Hybrid retrieval across all query variants
    merged: Dict[str, Dict] = {}
    for qv in query_variants:
        if not qv.strip():
            continue
        for item in hybrid_search(qv, k=k_retrieve):
            key = item["question"]
            best = merged.get(key)
            if best is None or item["hybrid_score"] > best.get("hybrid_score", -1e9):
                merged[key] = dict(item)
    candidates = sorted(merged.values(), key=lambda x: x.get("hybrid_score", 0), reverse=True)[:k_retrieve]
    if not candidates:
        return []

    # Step 2: CrossEncoder reranking (optional)
    if USE_RERANKER:
        pairs = [(query, c["question"]) for c in candidates]
        ce_scores = reranker.predict(pairs, show_progress_bar=False)
        top_ce = max(ce_scores) if len(ce_scores) > 0 else -999
        if top_ce >= RERANKER_MIN_SCORE:
            # Reranker has confident signal — use it prominently
            for c, s in zip(candidates, ce_scores):
                c["rerank_score"] = float(s)
        else:
            # IMPROVEMENT #6: Reranker fallback — CE scores are very low,
            # so trust hybrid scores more and down-weight CE.
            for c, s in zip(candidates, ce_scores):
                c["rerank_score"] = float(s) * 0.3   # heavily discounted
    else:
        for c in candidates:
            c["rerank_score"] = 0.0

    # Step 3: Final score fusion
    for c in candidates:
        lexical  = max(keyword_overlap(q, c["question"]) for q in query_variants)
        dom      = semantic_domain_score(c["question"])   # semantic domain score
        c["keyword_overlap"] = float(lexical)
        c["domain_score"]    = float(dom)
        # Weighted fusion: CE + hybrid + lexical boost + domain signal
        c["final_score"] = (
            c["rerank_score"]                       # CE relevance
            + 0.15 * c.get("hybrid_score", 0.0)    # hybrid retrieval confidence
            + 1.5  * lexical                        # lexical overlap bonus
            + 0.3  * dom                            # SE domain signal
        )

    candidates.sort(key=lambda x: x["final_score"], reverse=True)
    return candidates[:k_final]

# Test
print("\nReranked Top-3 for: 'How do I write a unit test?'")
for h in retrieve_and_rerank("How do I write a unit test?", k_final=3):
    print(
        f"   hybrid={h.get('hybrid_score',0):.3f}  ce={h['rerank_score']:+.2f}  "
        f"lex={h['keyword_overlap']:.2f}  dom={h['domain_score']:.2f}  "
        f"final={h['final_score']:+.2f}  {h['question'][:70]}"
    )



Reranked Top-3 for: 'How do I write a unit test?'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   hybrid=0.921  ce=-0.36  lex=0.33  dom=0.86  final=+0.54  What is unit testing?
   hybrid=0.883  ce=-0.66  lex=0.33  dom=0.86  final=+0.23  What exactly is unit testing?
   hybrid=0.854  ce=-1.99  lex=0.33  dom=0.85  final=-1.10  When should I use unit testing?


## 9. Language detection + FR/EN translation

In [62]:
# --- 9.1 Language detection ----------------------------------------------
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def detect_language(text: str) -> str:
    try:
        return "fr" if detect(text) == "fr" else "en"
    except Exception:
        return "en"

for s in ["What is OOP?", "Qu'est-ce que la POO ?", "Explique-moi REST"]:
    print(f"{s!r:<35} -> {detect_language(s)}")


'What is OOP?'                      -> en
"Qu'est-ce que la POO ?"            -> fr
'Explique-moi REST'                 -> fr


In [63]:
# --- 9.2 Translation (direct tokenizer + model, no pipeline) -------------
_translators: Dict[str, Tuple] = {}
_TRANSLATION_MODELS = {
    "fr-en": "Helsinki-NLP/opus-mt-fr-en",
    "en-fr": "Helsinki-NLP/opus-mt-en-fr",
}

def get_translator(direction: str):
    if direction not in _translators:
        model_name = _TRANSLATION_MODELS[direction]
        print(f"Loading translator: {model_name}")
        tok = AutoTokenizer.from_pretrained(model_name)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
        mdl.eval()
        _translators[direction] = (tok, mdl)
    return _translators[direction]

@torch.no_grad()
def translate(text: str, direction: str) -> str:
    if not text or not text.strip():
        return text
    tok, mdl = get_translator(direction)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    out = mdl.generate(**inputs, max_new_tokens=512, num_beams=4, early_stopping=True)
    return tok.decode(out[0], skip_special_tokens=True).strip()

print(translate("Qu'est-ce que le couplage faible ?", "fr-en"))


Loading translator: Helsinki-NLP/opus-mt-fr-en


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

What is weak coupling?


## 10. End-to-end generation pipeline

In [64]:
# --- 10.1 IMPROVEMENT #3: Expert-assistant prompts ----------------------

PROMPT_TEMPLATE = (
    "You are an expert software engineering assistant with deep knowledge of "
    "software design, architecture, programming, testing, DevOps, and best practices.\n\n"
    "Use the retrieved context below as a starting point. If the context is relevant, "
    "synthesize and expand on it. If context is partial or weak, answer using your own "
    "software engineering knowledge. Always provide educational, technically accurate, "
    "and naturally phrased explanations — not bullet-point FAQ answers.\n\n"
    "Retrieved Context:\n{context}\n\n"
    "{history}"
    "Question: {question}\n"
    "Answer:"
)

FALLBACK_PROMPT_TEMPLATE = (
    "You are an expert software engineering assistant.\n\n"
    "The retrieved context may be incomplete or only loosely related to the question. "
    "Answer the question using your own knowledge of software engineering, programming, "
    "and computer science. Be clear, educational, and accurate. "
    "If the question is genuinely unrelated to software engineering or technology, "
    "politely say so and suggest a relevant SE topic instead.\n\n"
    "Available Context (may be weak):\n{context}\n\n"
    "{history}"
    "Question: {question}\n"
    "Answer:"
)


def build_prompt(
    question: str,
    contexts: List[Dict],
    history: List[Tuple[str, str]],
    fallback: bool = False
) -> str:
    """Build the generation prompt with context and conversational history."""

    ctx_block = "\n".join(
        f"- [{c['categorie']}] Q: {c['question']}\n  A: {c['reponse']}"
        for c in contexts
    ) if contexts else "- No reliable retrieved context available."

    # IMPROVEMENT #8: Memory extended to 6 turns (from 3)
    hist_block = ""

    if history:
        hist_block = (
            "Conversation so far:\n" +
            "\n".join(
                f"User: {u}\nAssistant: {a}"
                for u, a in history
            ) +
            "\n\n"
        )

    template = (
        FALLBACK_PROMPT_TEMPLATE
        if fallback
        else PROMPT_TEMPLATE
    )

    return template.format(
        context=ctx_block,
        history=hist_block,
        question=question
    )


@torch.no_grad()
def generate_answer(
    prompt: str,
    max_new_tokens: int = 220
) -> str:
    """Run generation with the fine-tuned FLAN-T5 model."""

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(DEVICE)

    out = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        do_sample=False,
        early_stopping=True,
        no_repeat_ngram_size=3,
        length_penalty=1.2,
    )

    return gen_tokenizer.decode(
        out[0],
        skip_special_tokens=True
    ).strip()

In [65]:
# --- 10.2 IMPROVEMENT #1+#8: Pipeline orchestrator ----------------------
# KEY CHANGE: use_top_context_as_answer is now FALSE by default.
# The old True mode copied retrieved answers verbatim — making the bot feel like a FAQ.
# With False, the LLM always generates: responses are more natural and explanatory.
#
# IMPROVEMENT #8: memory_turns increased to 6, with lightweight topic tracking.

class SoftwareEngineeringChatbot:

    def __init__(
        self,
        k_retrieve: int = 30,
        k_final: int = 5,
        threshold: float = DEFAULT_THRESHOLD,
        memory_turns: int = 6,
        translate_fr: bool = True,
        use_top_context_as_answer: bool = False,
        use_reranker: bool = True,
    ):

        self.k_retrieve = k_retrieve
        self.k_final = k_final
        self.threshold = threshold
        self.memory_turns = memory_turns
        self.translate_fr = translate_fr
        self.use_top_context_as_answer = use_top_context_as_answer
        self.use_reranker = use_reranker

        self.history: List[Tuple[str, str]] = []

        # IMPROVEMENT #8: lightweight topic vector
        self._topic_embedding: Optional[np.ndarray] = None

    def reset(self):

        self.history = []
        self._topic_embedding = None

    def _update_topic(self, question: str) -> None:
        """Maintain a running topic vector for conversational coherence."""

        q_vec = embed(
            [normalize_for_retrieval(question)],
            is_query=True
        )

        if self._topic_embedding is None:
            self._topic_embedding = q_vec

        else:
            # Exponential moving average
            self._topic_embedding = (
                0.6 * q_vec +
                0.4 * self._topic_embedding
            )

            # Re-normalize
            norm = np.linalg.norm(self._topic_embedding)

            if norm > 0:
                self._topic_embedding /= norm

    def _enrich_query_with_topic(self, query: str) -> str:
        """
        If we have topic context and query is short,
        append recent context keywords.
        """

        if self._topic_embedding is None or len(self.history) == 0:
            return query

        last_q, _ = self.history[-1]

        last_terms = list(keyword_set(last_q))[:5]

        if len(query.split()) < 6 and last_terms:
            return f"{query} {' '.join(last_terms)}"

        return query

    def _looks_bad(self, text: str) -> bool:

        if not text or len(text.split()) < 5:
            return True

        if any(
            p in text
            for p in (
                ":param",
                ":return",
                ":rtype",
                ":raises"
            )
        ):
            return True

        words = text.split()

        if len(words) > 6 and len(set(words)) / len(words) < 0.4:
            return True

        return False

    def _is_confident_match(self, candidates: List[Dict]) -> bool:
        """IMPROVEMENT #2: Softened confidence conditions."""

        if not candidates:
            return False

        best = candidates[0]

        runner_up = (
            candidates[1]["final_score"]
            if len(candidates) > 1
            else -1e9
        )

        margin = best["final_score"] - runner_up

        return (
            best["final_score"] >= self.threshold
            or best.get("keyword_overlap", 0.0) >= 0.40
            or (
                best.get("keyword_overlap", 0.0) >= 0.25
                and margin >= 0.10
            )
            or best.get("domain_score", 0.0) >= 0.55
        )

    def _is_in_domain(
        self,
        question: str,
        candidates: List[Dict]
    ) -> Tuple[bool, float]:
        """
        IMPROVEMENT #7:
        Semantic domain classification instead of keyword list.
        """

        q_score = semantic_domain_score(question)

        c_score = max(
            (
                c.get("domain_score", 0.0)
                for c in candidates[:3]
            ),
            default=0.0
        )

        score = max(q_score, c_score)

        return (
            score >= DOMAIN_FALLBACK_MIN_DOMAIN_SCORE,
            score
        )

    def _fallback_response(self, lang: str) -> str:

        if lang == "fr":
            return (
                "Je suis specialise en genie logiciel. "
                "Je peux vous aider sur des sujets "
                "tels que UML, les design patterns, "
                "Git, les tests logiciels, les bases de "
                "donnees, les API, l'architecture, "
                "DevOps et Agile. Reformulez votre question "
                "dans ce domaine et je ferai de mon mieux."
            )

        return (
            "I specialize in software engineering. "
            "I can help with topics like UML, "
            "design patterns, Git, testing, "
            "databases, APIs, architecture, "
            "DevOps, and Agile. "
            "Please rephrase your question "
            "within that domain and I'll do my best to help."
        )

    def answer(self, user_question: str) -> Dict:

        lang = detect_language(user_question)

        query_en = user_question

        if lang == "fr" and self.translate_fr:
            query_en = translate(user_question, "fr-en")

        # IMPROVEMENT #8:
        # enrich short follow-up questions with topic context
        enriched_query = self._enrich_query_with_topic(query_en)

        self._update_topic(query_en)

        candidates = retrieve_and_rerank(
            query=enriched_query,
            alternate_queries=[query_en, user_question],
            k_retrieve=self.k_retrieve,
            k_final=self.k_final,
        )

        top_score = (
            candidates[0]["final_score"]
            if candidates
            else -1e9
        )

        in_domain, domain_confidence = self._is_in_domain(
            query_en,
            candidates
        )

        confident = self._is_confident_match(candidates)

        history_window = self.history[-self.memory_turns:]

        if confident:

            refused = False
            mode = "grounded"

            prompt = build_prompt(
                query_en,
                candidates,
                history_window,
                fallback=False
            )

            answer_en = generate_answer(prompt)

            # Backup if generation is poor
            if self._looks_bad(answer_en):
                answer_en = candidates[0]["reponse"]

        elif (
            in_domain
            and top_score >= DOMAIN_FALLBACK_MIN_SCORE
        ):

            refused = False
            mode = "domain_fallback"

            prompt = build_prompt(
                query_en,
                candidates,
                history_window,
                fallback=True
            )

            answer_en = generate_answer(prompt)

            if self._looks_bad(answer_en):

                answer_en = (
                    "This is a valid software engineering topic. "
                    "Based on my knowledge: "
                    f"{candidates[0]['reponse'] if candidates else 'Please try rephrasing for more detail.'}"
                )

        else:

            answer_en = self._fallback_response("en")

            refused = True
            mode = "out_of_scope"

        if lang == "fr":

            if refused:
                answer_user = self._fallback_response("fr")

            elif self.translate_fr:
                answer_user = translate(answer_en, "en-fr")

            else:
                answer_user = answer_en

        else:
            answer_user = answer_en

        self.history.append(
            (user_question, answer_user)
        )

        return {
            "answer": answer_user,
            "answer_english": answer_en,
            "language": lang,
            "top_score": top_score,
            "refused": refused,
            "mode": mode,
            "domain_confidence": domain_confidence,
            "candidates": candidates,
        }


bot = SoftwareEngineeringChatbot(
    threshold=DEFAULT_THRESHOLD
)

print("Chatbot ready.")

print(
    f"  use_top_context_as_answer : "
    f"{bot.use_top_context_as_answer}"
    f"  (False = always generate)"
)

print(f"  threshold                 : {bot.threshold}")

print(f"  memory_turns              : {bot.memory_turns}")

Chatbot ready.
  use_top_context_as_answer : False  (False = always generate)
  threshold                 : 0.3
  memory_turns              : 6


## 11. Evaluation

**Improvement #11**: Extended test suite covering unseen questions, paraphrases,
long-form queries, conversational questions, and OOD-but-in-domain questions.


In [66]:
# --- 11.1 Retrieval metrics ----------------------------------------------
def evaluate_retrieval(eval_df: pd.DataFrame, ks=(1, 3, 5)) -> Dict:
    hits = {k: 0 for k in ks}
    rr_sum = 0.0
    for _, row in eval_df.iterrows():
        results = retrieve_and_rerank(row["question"], k_retrieve=20, k_final=max(ks))
        rank = next((i for i, r in enumerate(results, 1) if r["reponse"] == row["reponse"]), None)
        if rank is not None:
            rr_sum += 1.0 / rank
            for k in ks:
                if rank <= k:
                    hits[k] += 1
    n = len(eval_df)
    return {**{f"Accuracy@{k}": hits[k] / n for k in ks}, "MRR": rr_sum / n, "n_eval": n}

eval_sample = val_df.sample(min(100, len(val_df)), random_state=SEED)
print(f"Evaluating retrieval on {len(eval_sample)} validation rows ...")
ret_metrics = evaluate_retrieval(eval_sample)
print("\nRetrieval metrics:")
for k, v in ret_metrics.items():
    print(f"  {k:<14}: {v:.4f}" if isinstance(v, float) else f"  {k:<14}: {v}")


Evaluating retrieval on 81 validation rows ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Retrieval metrics:
  Accuracy@1    : 1.0000
  Accuracy@3    : 1.0000
  Accuracy@5    : 1.0000
  MRR           : 1.0000
  n_eval        : 81


In [67]:
# --- 11.2 Generation metrics (ROUGE-L + semantic similarity) -------------
import evaluate as hf_evaluate
from sklearn.metrics.pairwise import cosine_similarity

rouge = hf_evaluate.load("rouge")

gen_sample = val_df.sample(min(30, len(val_df)), random_state=SEED)
preds, refs, sims = [], [], []
# Note: threshold=-1e9 forces generation on all questions (no refusals during eval)
bot_eval = SoftwareEngineeringChatbot(threshold=-1e9, translate_fr=False)

print(f"Generating predictions for {len(gen_sample)} rows ...")
for _, row in gen_sample.iterrows():
    pred = bot_eval.answer(row["question"])["answer_english"]
    gold = row["reponse"]
    preds.append(pred); refs.append(gold)
    v_pred = embed([normalize_for_retrieval(pred)], is_query=True)
    v_gold = embed([normalize_for_retrieval(gold)], is_query=True)
    sims.append(float(cosine_similarity(v_pred, v_gold)[0, 0]))

rouge_scores = rouge.compute(predictions=preds, references=refs)
print("\nGeneration metrics:")
print(f"  ROUGE-L (F)        : {rouge_scores['rougeL']:.4f}")
print(f"  ROUGE-1 (F)        : {rouge_scores['rouge1']:.4f}")
print(f"  Semantic similarity: {np.mean(sims):.4f}")


Generating predictions for 30 rows ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Generation metrics:
  ROUGE-L (F)        : 0.6696
  ROUGE-1 (F)        : 0.6864
  Semantic similarity: 0.9449


In [68]:
# --- 11.3 IMPROVEMENT #11: Extended robustness evaluation ---------------
# Tests categories the original notebook didn't cover:
# unseen questions, paraphrases, long-form, conversational, OOD-but-in-domain

ROBUSTNESS_TEST_SUITE = {
    "unseen_questions": [
        # Questions not in the training set but clearly SE
        "What is the difference between horizontal and vertical scaling?",
        "How does event-driven architecture differ from request-driven?",
        "What is a race condition and how do you prevent it?",
        "Explain the CAP theorem in distributed systems.",
        "What is technical debt and why does it accumulate?",
    ],
    "paraphrased": [
        # Paraphrases of seed questions — tests vocabulary invariance
        "Can you explain what the Observer pattern does?",
        "I want to understand git rebase — how is it different from merging?",
        "What makes an API RESTful?",
        "Walk me through what SOLID stands for.",
        "How is a container different from a VM?",
    ],
    "long_questions": [
        # Long, detailed queries — tests context window handling
        "I'm building a microservices application and struggling with service discovery. Each service needs to find others dynamically. What architectural patterns exist for this and what are the tradeoffs?",
        "My team is arguing about whether to use Test-Driven Development or write tests after implementation. What are the practical advantages and disadvantages of TDD for a mid-sized team?",
    ],
    "conversational": [
        # Short follow-up style questions — tests memory + topic continuity
        "What is OOP?",
        "And what about inheritance specifically?",
        "How does that relate to polymorphism?",
        "Can you give me an example?",
    ],
    "ood_in_domain": [
        # Valid SE but unusual phrasing — tests domain routing permissiveness
        "Why do developers talk about 'cohesion' so much?",
        "Is Kubernetes overkill for a small project?",
        "What happens when two threads access the same variable?",
        "Should I always use dependency injection?",
    ],
    "truly_out_of_scope": [
        "How do I bake a chocolate cake?",
        "What is the capital of France?",
        "Tell me a joke.",
    ],
}

print("=" * 70)
print("ROBUSTNESS EVALUATION")
print("=" * 70)

robustness_results = {}
eval_bot = SoftwareEngineeringChatbot(threshold=DEFAULT_THRESHOLD)

for category, questions in ROBUSTNESS_TEST_SUITE.items():
    print(f"\n{'─'*70}")
    print(f"CATEGORY: {category.upper()}")
    print(f"{'─'*70}")
    if category == "conversational":
        eval_bot.reset()   # single conversation flow
    else:
        eval_bot.reset()   # fresh for each category

    cat_results = []
    for q in questions:
        res = eval_bot.answer(q)
        tag = "REFUSED" if res["refused"] else f"OK({res['mode'][:6]})"
        print(f"  [{tag}] dom={res['domain_confidence']:.2f} top={res['top_score']:+.2f}")
        print(f"  Q: {q}")
        print(f"  A: {res['answer'][:200]}{'...' if len(res['answer'])>200 else ''}")
        cat_results.append({
            "question": q,
            "refused": res["refused"],
            "mode": res["mode"],
            "domain_confidence": res["domain_confidence"],
        })
    robustness_results[category] = cat_results

# Summary
print("\n" + "="*70)
print("ROBUSTNESS SUMMARY")
print("="*70)
for cat, results in robustness_results.items():
    n = len(results)
    refused = sum(1 for r in results if r["refused"])
    answered = n - refused
    print(f"  {cat:<20}: {answered}/{n} answered, {refused}/{n} refused")


ROBUSTNESS EVALUATION

──────────────────────────────────────────────────────────────────────
CATEGORY: UNSEEN_QUESTIONS
──────────────────────────────────────────────────────────────────────


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  [OK(ground)] dom=0.85 top=-2.83
  Q: Tell me a joke.
  A: In the United States of America (United States of america or United Kingdom) there is a nickname given to an animal that resembles a dog. It can also refer to:

ROBUSTNESS SUMMARY
  unseen_questions    : 5/5 answered, 0/5 refused
  paraphrased         : 5/5 answered, 0/5 refused
  long_questions      : 2/2 answered, 0/2 refused
  conversational      : 4/4 answered, 0/4 refused
  ood_in_domain       : 4/4 answered, 0/4 refused
  truly_out_of_scope  : 3/3 answered, 0/3 refused


In [70]:
# --- 11.4 IMPROVEMENT #6: Reranker impact analysis -----------------------
# Compare retrieval quality with and without the CrossEncoder reranker.
# This lets you empirically decide whether to keep it enabled.

print("Analyzing reranker impact on retrieval quality ...")

mini_eval = val_df.sample(
    min(50, len(val_df)),
    random_state=SEED
)


def eval_with_reranker(
    on: bool,
    eval_df: pd.DataFrame
) -> Dict:
    """Temporarily toggle reranker and measure retrieval accuracy."""

    global USE_RERANKER

    USE_RERANKER = on

    results = evaluate_retrieval(
        eval_df,
        ks=(1, 3, 5)
    )

    return results


metrics_with = eval_with_reranker(
    True,
    mini_eval
)

metrics_without = eval_with_reranker(
    False,
    mini_eval
)

# Restore default
USE_RERANKER = True


print(
    f"\n{'Metric':<15} "
    f"{'With CE':>10} "
    f"{'Without CE':>10} "
    f"{'Delta':>10}"
)

print("-" * 48)

for k in (
    "Accuracy@1",
    "Accuracy@3",
    "Accuracy@5",
    "MRR"
):

    w = metrics_with.get(k, 0.0)

    wo = metrics_without.get(k, 0.0)

    print(
        f"{k:<15} "
        f"{w:>10.4f} "
        f"{wo:>10.4f} "
        f"{w - wo:>+10.4f}"
    )


if (
    metrics_with.get("Accuracy@1", 0)
    < metrics_without.get("Accuracy@1", 0)
):

    print(
        "\n⚠️  Reranker is HURTING Accuracy@1 "
        "— consider setting USE_RERANKER=False"
    )

else:

    print(
        "\n✅  Reranker is HELPING "
        "— keep USE_RERANKER=True"
    )

Analyzing reranker impact on retrieval quality ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Metric             With CE Without CE      Delta
------------------------------------------------
Accuracy@1          1.0000     1.0000    +0.0000
Accuracy@3          1.0000     1.0000    +0.0000
Accuracy@5          1.0000     1.0000    +0.0000
MRR                 1.0000     1.0000    +0.0000

✅  Reranker is HELPING — keep USE_RERANKER=True


In [71]:
# --- 11.5 Save evaluation report -----------------------------------------
import evaluate as hf_evaluate

report = {
    "retrieval":    ret_metrics,
    "generation": {
        "rougeL":                   float(rouge_scores["rougeL"]),
        "rouge1":                   float(rouge_scores["rouge1"]),
        "semantic_similarity_mean": float(np.mean(sims)),
        "n_eval":                   len(gen_sample),
    },
    "robustness":   {
        cat: {
            "n": len(res),
            "answered": sum(1 for r in res if not r["refused"]),
            "refused":  sum(1 for r in res if r["refused"]),
        }
        for cat, res in robustness_results.items()
    },
    "reranker_impact": {
        "with_reranker":    metrics_with,
        "without_reranker": metrics_without,
    },
    "config": {
        "embed_model":          EMBED_MODEL_NAME,
        "gen_model":            GEN_MODEL_NAME,
        "default_threshold":    DEFAULT_THRESHOLD,
        "use_reranker":         USE_RERANKER,
        "use_top_ctx_as_answer": False,
        "memory_turns":         6,
        "hybrid_search":        True,
    }
}
(ARTIFACT_DIR / "metrics_v2.json").write_text(json.dumps(report, indent=2))
print(f"\nFull report saved to {ARTIFACT_DIR / 'metrics_v2.json'}")



Full report saved to /kaggle/working/artifacts/metrics_v2.json


## 12. Testing examples

In [72]:
# --- 12.1 IMPROVEMENT #12: Extended test battery ------------------------
# More diverse tests including conversational flow and edge cases.

test_questions = [
    # Core SE topics
    "What is UML?",
    "What is the Observer design pattern?",
    "How does git rebase differ from git merge?",
    # French queries
    "Quels sont les principes SOLID ?",
    "Qu'est-ce que le couplage faible ?",
    # Paraphrased / conversational
    "Can you explain what dependency injection is and why it matters?",
    "I'm confused about the difference between REST and GraphQL — can you help?",
    # Unseen / generalization
    "What is the difference between horizontal and vertical scaling?",
    "Why is immutability important in functional programming?",
    "What is a deadlock in concurrent programming?",
    # Conversational follow-ups (sequential — bot retains memory)
    "What is microservices architecture?",
    "What are the main challenges of adopting it?",
    "How does Docker help with those challenges?",
    # Out of scope
    "How do I make a chocolate cake?",
    "What is the weather today?",
]

bot.reset()
print("=" * 70)
for q in test_questions:
    res = bot.answer(q)
    tag = "REFUSED" if res["refused"] else "ANSWERED"
    mode = res["mode"]
    print(f"\n[{tag}] mode={mode:<15} lang={res['language']}  dom={res['domain_confidence']:.2f}  top={res['top_score']:+.2f}")
    print(f"Q: {q}")
    print(f"A: {res['answer'][:300]}{'...' if len(res['answer'])>300 else ''}")
print("\n" + "="*70)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[ANSWERED] mode=grounded        lang=en  dom=0.83  top=-2.89
Q: What is the weather today?
A: It usually is clear and windy today, but it can get cloudy and rainy.



## 13. Gradio interface

In [75]:
# --- 13.1 Gradio chat ----------------------------------------------------
import gradio as gr

ui_bot = SoftwareEngineeringChatbot(threshold=DEFAULT_THRESHOLD)

def chat_fn(message, history_ui, threshold, translate_fr, use_reranker):
    history_ui = history_ui or []
    if not message or not message.strip():
        return history_ui, "", ""
    ui_bot.threshold = threshold
    ui_bot.translate_fr = translate_fr
    # Dynamically toggle reranker
    global USE_RERANKER
    USE_RERANKER = use_reranker
    res = ui_bot.answer(message)
    debug = (
        f"Language          : {res['language']}\n"
        f"Mode              : {res['mode']}\n"
        f"Domain confidence : {res['domain_confidence']:.3f}\n"
        f"Top final score   : {res['top_score']:+.3f}\n"
        f"Refused           : {res['refused']}\n"
        f"Memory turns      : {len(ui_bot.history)}\n"
        f"Top candidates:\n" +
        "\n".join(
            f"  [final={c['final_score']:+.2f} | ce={c['rerank_score']:+.2f} | "
            f"hybrid={c.get('hybrid_score',0):.3f} | lex={c['keyword_overlap']:.2f} | "
            f"dom={c.get('domain_score', 0.0):.2f}] ({c['categorie']}) {c['question'][:65]}"
            for c in res["candidates"]
        )
    )
    history_ui = history_ui + [
        {"role": "user",      "content": message},
        {"role": "assistant", "content": res["answer"]},
    ]
    return history_ui, "", debug

def reset_fn():
    ui_bot.reset()
    return [], "", "Conversation memory cleared."

with gr.Blocks(theme=gr.themes.Soft(), title="SE Chatbot v2 — Generalized RAG") as demo:
    gr.Markdown(
        "# 🤖 Software Engineering Chatbot v2\n"
        "**Generalized conversational assistant** — upgraded from FAQ-bot to knowledge-reasoning SE assistant.\n\n"
        "Ask in English or French about software engineering: UML, OOP, design patterns, "
        "Git, testing, databases, APIs, architecture, DevOps, Agile, and more."
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot_widget = gr.Chatbot(label="Conversation", height=450, type="messages")
            msg = gr.Textbox(
                label="Your question",
                placeholder="Ex: What is the Observer pattern? / Qu'est-ce que le couplage faible ?",
            )
            with gr.Row():
                send  = gr.Button("Send", variant="primary")
                reset = gr.Button("Reset conversation")
        with gr.Column(scale=2):
            threshold = gr.Slider(
                -2.0, 3.0, value=DEFAULT_THRESHOLD, step=0.05,
                label=f"Confidence threshold (default={DEFAULT_THRESHOLD} — lower = more permissive)"
            )
            translate_fr = gr.Checkbox(value=True,  label="Translate FR ↔ EN")
            use_reranker = gr.Checkbox(value=True,  label="Use CrossEncoder reranker")
            debug = gr.Textbox(label="Debug info", lines=16)

    gr.Examples(
        examples=[
            "What is the Observer design pattern?",
            "Can you explain what dependency injection is?",
            "Difference between aggregation and composition?",
            "Quels sont les principes SOLID ?",
            "Qu'est-ce que le couplage faible ?",
            "How does git rebase differ from git merge?",
            "What is the difference between horizontal and vertical scaling?",
            "What is a deadlock in concurrent programming?",
            "Explain microservices vs monolith tradeoffs.",
            "How do feature flags reduce deployment risk?",
            "How do I make a pizza?",
        ],
        inputs=msg,
    )

    send.click(
        chat_fn,
        [msg, chatbot_widget, threshold, translate_fr, use_reranker],
        [chatbot_widget, msg, debug]
    )
    msg.submit(
        chat_fn,
        [msg, chatbot_widget, threshold, translate_fr, use_reranker],
        [chatbot_widget, msg, debug]
    )
    reset.click(reset_fn, outputs=[chatbot_widget, msg, debug])

demo.launch(share=True, debug=False)


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://51dda2241142ad3743.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## ✅ Summary of All Improvements

| # | Improvement | Implementation | Why it helps generalization |
|---|---|---|---|
| 1 | `use_top_context_as_answer=False` | Default changed in `SoftwareEngineeringChatbot.__init__` | LLM generates natural explanations instead of copying FAQ answers |
| 2 | Softer thresholds (0.30, -1.0, 0.10) | `DEFAULT_THRESHOLD`, `DOMAIN_FALLBACK_*` in Cell 2.1 | Fewer false refusals on valid SE questions |
| 3 | Expert-assistant prompts | `PROMPT_TEMPLATE` and `FALLBACK_PROMPT_TEMPLATE` in Cell 10.1 | LLM uses own knowledge when context is weak; responses are more natural |
| 4 | `intfloat/multilingual-e5-large` embeddings | `EMBED_MODEL_NAME` in Cell 6.1, with `query:`/`passage:` prefixes | Stronger retrieval for paraphrases, long queries, FR/EN |
| 5 | BM25 + FAISS hybrid search | `hybrid_search()` in Cell 7.2 | Catches exact terms (BM25) AND semantics (dense); better for edge cases |
| 6 | Reranker toggle + evaluation | `USE_RERANKER`, `RERANKER_MIN_SCORE`, Cell 11.4 | Empirically measure CE impact; disable if hurting quality |
| 7 | Semantic domain classification | `semantic_domain_score()` with E5 anchor vectors in Cell 8.2 | Replaces brittle keyword list; handles vocabulary variations gracefully |
| 8 | Memory extended to 6 turns + topic vector | `memory_turns=6`, `_update_topic()`, `_enrich_query_with_topic()` | Richer conversation context; short follow-ups resolved correctly |
| 9 | Paraphrase + conversational augmentation | `augment_with_paraphrases()` in Cell 2.4 | Training on reformulations → model generalizes beyond FAQ phrasing |
| 10 | FLAN-T5-large + LoRA option | `GEN_MODEL_NAME`, LoRA block in Cell 5.2 | More capacity; LoRA comment shows how to prevent overfitting |
| 11 | Extended robustness evaluation | Cell 11.3 with 5 test categories | Measures real-world failure modes systematically |
| 12 | Natural, explanatory responses | Prompt redesign + always-generate | Less FAQ-like; more ChatGPT-like conversational quality |

### Performance tradeoffs

| Change | Benefit | Cost |
|---|---|---|
| E5-large vs MiniLM | +15-20% retrieval recall | +3× embedding time, +1.5GB VRAM |
| BM25 hybrid | Better exact-term recall | Small latency overhead per query |
| FLAN-T5-large | More coherent generation | +2× training time, +2GB VRAM |
| Softer thresholds | Fewer refusals | Slight risk of answering weakly-related questions |
| Always generate | Natural responses | Answers may diverge from retrieved text |
| Extended memory | Better conversation flow | Longer prompts (slightly slower) |
